In [3]:
import os
import json
import math
import pandas as pd

# --- Configuración de rutas de directorios ---
results_dir = "data_for_seeds"

# Escanear y filtrar archivos solo para n=33
json_files = [
    f for f in os.listdir(results_dir) 
    if f.startswith("simulation_data_") and "_n33_" in f and f.endswith(".json")
]

rows_accumulator = []

for current_file in sorted(json_files):
    file_path = os.path.join(results_dir, current_file)
    
    with open(file_path, "r") as f:
        payload = json.load(f)
    
    meta = payload["meta_parameters"]
    beta_val = meta["beta"]
    m_val = meta["m"]
    n_val = meta["n"]
    
    for instance in payload["results"]:
        instance_id = instance["instance_id"]
        
        # Recuperar estructuras nativas del JSON
        sorted_cols = instance["sorted_cols"]
        posteriors = instance["posteriors"]
        posterior_matrix = instance["posterior_matrix"]  # Tamaño: m x n
        E_tilde_matrix = instance["E_tilde_matrix"]      # Tamaño: m x n
        
        # =====================================================================
        # TAREA 1: Procesamiento por Columnas Completas (Top 8 MAP)
        # =====================================================================
        # Tomar los mejores 8 índices de columnas de sorted_cols (ya en orden descendente)
        top_8_cols = sorted_cols[:8]
        
        col_bit_strings = []
        global_col_score = 1.0
        
        for col_idx in top_8_cols:
            col_str_key = str(col_idx)
            score, col_raw_str = posteriors[col_str_key]
            
            # Limpiar el string "[1, 0, 1]" para transformarlo en "101"
            clean_bits = col_raw_str.replace("[", "").replace("]", "").replace(",", "").replace(" ", "")
            col_bit_strings.append(clean_bits)
            
            # Multiplicar verosimilitudes para el score global
            global_col_score *= float(score)
            
        # Concatenar todos los bits planos seguidos
        concatenated_cols_str = "".join(col_bit_strings)
        
        # =====================================================================
        # TAREA 2: Procesamiento por Entradas de Bits Individuales (Top 256)
        # =====================================================================
        # Construir una lista con todas las (m * n) entradas para poder ordenarlas
        all_entries = []
        for i in range(m_val):        # Filas (0 a m-1)
            for j in range(n_val):    # Columnas (0 a n-1)
                bit_score = posterior_matrix[i][j]
                observed_bit = int(E_tilde_matrix[i][j])
                all_entries.append(((i, j), bit_score, observed_bit))
                
        # Ordenar todas las celdas de mayor a menor probabilidad condicional (score)
        all_entries_sorted = sorted(all_entries, key=lambda x: x[1], reverse=True)
        
        # Tomar exactamente las mejores 256 entradas
        top_256_entries = all_entries_sorted[:256]
        
        entry_indices = []
        entry_bits = []
        global_entry_score = 1.0
        
        for (i, j), score, obs_bit in top_256_entries:
            entry_indices.append((i, j))
            entry_bits.append(str(obs_bit))
            global_entry_score *= float(score)
            
        concatenated_entries_str = "".join(entry_bits)
        
        # =====================================================================
        # Consolidar datos en el registro para el DataFrame
        # =====================================================================
        record = {
            "beta": beta_val,
            "instance_id": instance_id,
            "n": n_val,
            "k": meta["k"],
            "r": meta["r"],
            "m": m_val,
            
            # Columnas completas (Métricas de salida Caso 1)
            "col_top_8_indices": top_8_cols,
            "col_concatenated_string": concatenated_cols_str,
            "col_global_score": global_col_score,
            
            # Bits individuales (Métricas de salida Caso 2)
            "entry_top_256_indices": entry_indices,
            "entry_concatenated_string": concatenated_entries_str,
            "entry_global_score": global_entry_score
        }
        rows_accumulator.append(record)

# Crear el DataFrame de Pandas final
df_analysis = pd.DataFrame(rows_accumulator)

# --- Verificación en Consola ---
print(f"[SUCCESS] DataFrame generado. Total filas: {len(df_analysis)}")
print("\nEjemplo de la primera fila procesada:")
print(df_analysis[["beta", "instance_id", "col_global_score", "entry_global_score"]].head(1))

[SUCCESS] DataFrame generado. Total filas: 250

Ejemplo de la primera fila procesada:
   beta  instance_id  col_global_score  entry_global_score
0  0.03            1          0.013691            0.035055


In [4]:
df_analysis[["beta", "instance_id", "col_global_score", "entry_global_score", "col_concatenated_string", "entry_concatenated_string", "col_top_8_indices", "entry_top_256_indices"]].head()

,beta,instance_id,col_global_score,entry_global_score,col_concatenated_string,entry_concatenated_string,col_top_8_indices,entry_top_256_indices
0,0.03,1,0.013691,0.035055,0010010000011010100101110100001011000010001011...,0000000000000000000000000000000000000000000000...,"[17, 7, 32, 2, 1, 9, 22, 23]","[(0, 2), (0, 4), (0, 7), (0, 9), (0, 10), (0, ..."
1,0.03,2,0.013550,0.035055,1010010101010001000000011000001000001001110010...,0000000000000000000000000000000000000000000000...,"[17, 12, 31, 10, 15, 2, 20, 4]","[(0, 0), (0, 2), (0, 3), (0, 4), (0, 6), (0, 9..."
2,0.03,3,0.014123,0.035055,1100100100110001000000010001001100100110010010...,0000000000000000000000000000000000000000000000...,"[18, 29, 8, 32, 0, 13, 19, 4]","[(0, 0), (0, 2), (0, 3), (0, 4), (0, 5), (0, 7..."
3,0.03,4,0.014872,0.035055,0010100000000100010101001011000001100010110000...,0000000000000000000000000000000000000000000000...,"[2, 6, 4, 21, 1, 24, 9, 28]","[(0, 2), (0, 3), (0, 4), (0, 6), (0, 16), (0, ..."
4,0.03,5,0.013691,0.035055,0011011010100101000001000000011100100100000000...,0000000000000000000000000000000000000000000000...,"[4, 18, 8, 9, 27, 29, 31, 22]","[(0, 1), (0, 4), (0, 6), (0, 9), (0, 11), (0, ..."
